# 3.1.2 — Cartan ilişkilerinin 0-form ve 1-form üzerinde ispatı

**Hedef.** Yedi temel Cartan ilişkisinin tamamını sırayla bir 0-form $f$
üzerinde, ardından bir 1-form $\omega$ üzerinde sistemin içinde
adım-adım kapatmak.

| # | İlişki | Anlamı |
|---|---|---|
| 1 | $\iota_X \iota_Y + \iota_Y \iota_X = 0$ | iç çarpımlar anti-commute |
| 2 | $(\iota_X d + d \iota_X)\,\omega = \mathcal{L}_X \omega$ | Cartan'ın sihirli formülü |
| 3 | $[\mathcal{L}_X, \iota_Y] = \iota_{[X,Y]_{VF}}$ | Lie/iç çarpım komütatörü |
| 4 | $[d, \mathcal{L}_X] = 0$ | $d$ ve $\mathcal{L}_X$ commute |
| 5 | $d^2 = 0$ | dış türevin nilpotensi |
| 6 | $[\mathcal{L}_X, \mathcal{L}_Y] = \mathcal{L}_{[X,Y]_{VF}}$ | Lie türevlerinin komütatörü |

Her ilişki için iki alt-başlık göreceğiz: önce **0-form** $f$, sonra
**1-form** $\omega$. İspat zincirleri `display_chain` ile LaTeX olarak
basılır.

## Strateji

İki ayrı engine kuruyoruz; her biri ait olduğu form-derecesi için
doğal kuralları taşıyor.

| Form | Engine | Niçin |
|---|---|---|
| $f$ (0-form) | `default_engine` + 3 closure aksiyomu | $\iota_X(f) = 0$, $\mathcal{L}_X(f) = X(f)$, $\iota_X(df) = X(f)$, Cartan magic, $d^2 = 0$, $\iota_X^2 = 0$ — hepsi yerleşik. Closure aksiyomları $\mathcal{L}^2$, $[X,Y]$ kapanışını sağlıyor. |
| $\omega$ (1-form) | `intrinsic_engine_with_closure` + üç 0-form yardımcısı | İçinden geçen her ilişkiyi `multi_eval(\cdot, V_1, \dots, V_p)` üzerinden Koszul/Cartan açar; iç çarpımlar 0-form'a düştüğünde ek üç kural ($\iota_X(0\text{-form}) = 0$, $\iota_X^2 = 0$, $\mathcal{L}_X(0\text{-form}) = X(\cdot)$) bunları sıfırlar. |

İspat tek çağrıdan geçiyor: `prove_intrinsic_equivalence(LHS, RHS,
engine=…, registry=…)`. `Sum(LHS, -RHS)` üzerinde engine'i fix-point'e
kadar koşturuyor; sıfıra inerse `ProofChain`, kalırsa `ProofFailure`.

İlişki **(4)** $[\mathcal{L}_X, \iota_Y] = \iota_{[X,Y]_{VF}}$
1-form üzerinde her iki taraf 0-form skalere düşer; bu skalerleri
sistemin `multi_eval(\omega, Y) \equiv \omega(Y)` formuna açıkça
yazıyoruz. İntrinsik $\iota$/$\mathcal{L}$ kuralları bu şekil
üzerinde ateşliyor; *flow*-mode bir $\mathcal{L}_X$ kullanmak da
0-form skalere $\mathcal{L}_X(\omega(Y)) \to X(\omega(Y))$
indirgemesini açıyor.

In [2]:
# Notebook doğrudan açıldığında jacopy'ı import edilebilir hâle getirir.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

## 1. Kurulum — semboller, engineler, yardımcı fonksiyon

Aşağıda kullanılan semboller:

- $f$: bir 0-form (fonksiyon, `Graded(degree=0)`).
- $\omega$: bir 1-form (`Graded(degree=1)`).
- $X, Y, Z$: derece-0 vektör alanları (`Derivation`).

In [3]:
from jacopy.algebra.derivation import Act, Derivation
from jacopy.algebra.lie_bracket_vf import lie_bracket_vf
from jacopy.calculus.closure_axioms import (
    LieBracketVfAntiSymmetryDefinition,
    LieBracketVfJacobiDefinition,
    VfActCommutatorDefinition,
)
from jacopy.calculus.exterior_d import d as default_d
from jacopy.calculus.interior import interior
from jacopy.calculus.intrinsic_engine import (
    intrinsic_engine_with_closure,
    prove_intrinsic_equivalence,
)
from jacopy.calculus.lie_derivative import lie_derivative
from jacopy.core.expr import Integer, Neg, Sum, Symbol
from jacopy.core.multi_eval import multi_eval
from jacopy.core.properties import Graded
from jacopy.core.registry import PropertyRegistry
from jacopy.display.jupyter import display_chain
from jacopy.proof.expansion import (
    ExpansionEngine,
    IotaOnZeroFormDefinition,
    IotaSquaredZeroDefinition,
    LieDerivativeOnZeroFormDefinition,
    default_engine,
)

reg = PropertyRegistry()
f = Symbol("f");     reg.declare(f, Graded(degree=0))
omega = Symbol("ω"); reg.declare(omega, Graded(degree=1))
X = Derivation("X", 0)
Y = Derivation("Y", 0)
Z = Derivation("Z", 0)

print("f =", f, "  degree =", reg.get(f, Graded).degree)
print("ω =", omega, "  degree =", reg.get(omega, Graded).degree)

f = f   degree = 0
ω = ω   degree = 1


### 1.1. İki engine

**0-form engine.** `default_engine` zaten 0-form özelinde gerekli her
şeyi taşıyor: Cartan magic ($\mathcal{L}_X = d \circ \iota_X +
\iota_X \circ d$), $d^2 = 0$, $\iota_X^2 = 0$, $\iota_X(f) = 0$,
$\mathcal{L}_X(f) = X(f)$, $\iota_X(df) = X(f)$ ve operatör
distribüsyonu. Üç closure aksiyomu ekleniyor: $X(Y(f)) - Y(X(f)) =
[X,Y]_{VF}(f)$, $[X,Y]_{VF} + [Y,X]_{VF} = 0$ ve VF-Jacobi —
bunlar (7) numaralı ilişki için gereken tek katmandır.

**1-form engine.** `intrinsic_engine_with_closure` üç intrinsik kuralı
($\iota$, $\mathcal{L}$, $d$) + dört multi-eval yardımcısını + üç closure
aksiyomunu + `IotaActAsScalar` köprüsünü taşıyor. Buna üç adet *0-form
çöküş* kuralı ekleniyor: ilişki (1)/(2) için $\iota_Y(\omega)$ skaler
0-form'a düştüğünde $\iota_X$'in onu sıfırlaması, ve $\iota_X^2 = 0$
operatör formu.

In [4]:
# 0-form engine: default + closure aksiyomları
defs_zf = list(default_engine(registry=reg, d_squared_mode="axiom").definitions)
defs_zf += [
    VfActCommutatorDefinition(),
    LieBracketVfAntiSymmetryDefinition(),
    LieBracketVfJacobiDefinition(),
]
engine_zf = ExpansionEngine(defs_zf)

# 1-form engine: intrinsic + closure + 0-form çöküş kuralları
defs_of = list(intrinsic_engine_with_closure().definitions)
defs_of += [
    IotaOnZeroFormDefinition(registry=reg),
    IotaSquaredZeroDefinition(),
    LieDerivativeOnZeroFormDefinition(registry=reg),
]
engine_of = ExpansionEngine(defs_of)

print(f"0-form engine: {len(engine_zf.definitions)} kural")
print(f"1-form engine: {len(engine_of.definitions)} kural")

0-form engine: 11 kural
1-form engine: 14 kural


### 1.2. Yardımcı fonksiyon

Tek bir çağrı arayüzü: `prove(label, lhs, rhs, engine)`. Çıkış olarak
adım sayısını basıyor ve `ProofChain`'i döndürüyor; sonraki hücrelerde
`display_chain(chain)` zinciri LaTeX olarak gösteriyor.

In [5]:
def prove(label, lhs, rhs, engine):
    chain = prove_intrinsic_equivalence(lhs, rhs, engine=engine, registry=reg)
    print(f"{label} → {len(chain)} adımda kapandı")
    return chain

## 2. İlişki 1: iota anti-commute

$$
\iota_X \iota_Y + \iota_Y \iota_X = 0.
$$

### 2.1. 0-form $f$ üzerinde

$\iota_X(f) = 0$ olduğu için her iki terim ayrı ayrı sıfır;
toplam tabii ki sıfır.

In [6]:
lhs = Sum(Act(interior(X), Act(interior(Y), f)),
           Act(interior(Y), Act(interior(X), f)))
chain = prove("(0-form) ι_X ι_Y f + ι_Y ι_X f = 0", lhs, Integer(0), engine_zf)
display_chain(chain)

(0-form) ι_X ι_Y f + ι_Y ι_X f = 0 → 5 adımda kapandı


\begin{align*}
\iota_Y\!\left(f\right) &\to 0 && \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_X(f) = 0 on 0-forms} \\
\iota_X\!\left(0\right) &\to 0 && \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_X(f) = 0 on 0-forms} \\
\iota_X\!\left(f\right) &\to 0 && \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_X(f) = 0 on 0-forms} \\
\iota_Y\!\left(0\right) &\to 0 && \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_X(f) = 0 on 0-forms} \\
\left(0 + 0\right) - 0 &\to 0 && \text{[simplify]}\;\text{--- canonical-form pipeline (intra-loop)}
\end{align*}

### 2.2. 1-form üstü değerlendirme

$\omega$ bir 1-form olduğu için $\iota_Y(\omega)$ ve
$\iota_X(\omega)$ skaler 0-form'lardır; iç engine 0-form'a düşmüş
$\iota_X$'i kuralla sıfırlar.

In [7]:
lhs = Sum(Act(interior(X), Act(interior(Y), omega)),
           Act(interior(Y), Act(interior(X), omega)))
chain = prove("(1-form) ι_X ι_Y ω + ι_Y ι_X ω = 0", lhs, Integer(0), engine_of)
display_chain(chain)

(1-form) ι_X ι_Y ω + ι_Y ι_X ω = 0 → 3 adımda kapandı


\begin{align*}
\iota_X\!\left(\iota_Y\!\left(\omega\right)\right) &\to 0 && \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_X(f) = 0 on 0-forms} \\
\iota_Y\!\left(\iota_X\!\left(\omega\right)\right) &\to 0 && \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_X(f) = 0 on 0-forms} \\
\left(0 + 0\right) - 0 &\to 0 && \text{[simplify]}\;\text{--- canonical-form pipeline (intra-loop)}
\end{align*}

## 3. İlişki 3: Cartan'ın sihirli formülü

$$
(\iota_X \, d + d \, \iota_X)\,\omega \;=\; \mathcal{L}_X \omega.
$$

### 3.1. 0-form $f$ üzerinde

$f$ bir 0-form: $\iota_X(df) = X(f)$ ve $d(\iota_X f) = d(0) = 0$,
toplam $X(f) = \mathcal{L}_X f$.

In [10]:
lhs = Sum(Act(interior(X), Act(default_d, f)),
           Act(default_d, Act(interior(X), f)))
rhs = Act(lie_derivative(X), f)
chain = prove("(0-form) (ι_X d + d ι_X) f = L_X f", lhs, rhs, engine_zf)
display_chain(chain)

(0-form) (ι_X d + d ι_X) f = L_X f → 9 adımda kapandı


\begin{align*}
\iota_X\!\left(d\!\left(f\right)\right) &\to X\!\left(f\right) && \text{[\ensuremath{\iota}\_X(df) = X(f)]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_X(df) = X(f)} \\
\iota_X\!\left(f\right) &\to 0 && \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_X(f) = 0 on 0-forms} \\
L_X\!\left(f\right) &\to \left(d \, \iota_X\right)\!\left(f\right) + \left(\iota_X \, d\right)\!\left(f\right) && \text{[L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)]\,(axiom)}\;\text{--- apply axiom: L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)} \\
\left(X\!\left(f\right) + d\!\left(0\right)\right) - \left(\left(d \, \iota_X\right)\!\left(f\right) + \left(\iota_X \, d\right)\!\left(f\right)\right) &\to \left(X\!\left(f\right) + 0\right) - \left(d\!\left(\iota_X\!\left(f\right)\right) + \iota_X\!\left(d\!\left(f\right)\right)\right) && \text{[product-rule]}\;\text{--- graded Leibniz + linearity} \\
\left(X\!\left(f\right) + 0\right) - \left(d\!\left(\iota_X\!\left(f\right)\right) + \iota_X\!\left(d\!\left(f\right)\right)\right) &\to X\!\left(f\right) - d\!\left(\iota_X\!\left(f\right)\right) - \iota_X\!\left(d\!\left(f\right)\right) && \text{[simplify]}\;\text{--- canonical-form pipeline (intra-loop)} \\
\iota_X\!\left(f\right) &\to 0 && \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_X(f) = 0 on 0-forms} \\
\iota_X\!\left(d\!\left(f\right)\right) &\to X\!\left(f\right) && \text{[\ensuremath{\iota}\_X(df) = X(f)]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_X(df) = X(f)} \\
X\!\left(f\right) - d\!\left(0\right) - X\!\left(f\right) &\to X\!\left(f\right) - 0 - X\!\left(f\right) && \text{[product-rule]}\;\text{--- graded Leibniz + linearity} \\
X\!\left(f\right) - 0 - X\!\left(f\right) &\to 0 && \text{[simplify]}\;\text{--- canonical-form pipeline (intra-loop)}
\end{align*}

### 3.2. 1-form $\omega$ üzerinde, $Y$ ile değerlendir

1-form $\omega$ için her iki taraf 1-form'dur; ortak bir vektör
alanı $Y$ üzerinde değerlendirip Koszul açılımını çalıştırıyoruz.
İntrinsik kurallar bracket terimini ortadan kaldırarak iki tarafı
syntactic olarak eşitliyor.

In [11]:
lhs = Sum(multi_eval(Act(interior(X), Act(default_d, omega)), Y),
           multi_eval(Act(default_d, Act(interior(X), omega)), Y))
rhs = multi_eval(Act(lie_derivative(X), omega), Y)
chain = prove("(1-form, eval Y) ((ι_X d + d ι_X) ω)(Y) = (L_X ω)(Y)",
              lhs, rhs, engine_of)
display_chain(chain)

(1-form, eval Y) ((ι_X d + d ι_X) ω)(Y) = (L_X ω)(Y) → 6 adımda kapandı


\begin{align*}
\left(\iota_X\!\left(d\!\left(\omega\right)\right)\right)\!\left(Y\right) &\to \left(d\!\left(\omega\right)\right)\!\left(X,\, Y\right) && \text{[\ensuremath{\iota}\_X intrinsic: (\ensuremath{\iota}\_X \ensuremath{\omega})(Y\_1, …) = \ensuremath{\omega}(X, Y\_1, …)]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_X intrinsic: (\ensuremath{\iota}\_X \ensuremath{\omega})(Y\_1, …) = \ensuremath{\omega}(X, Y\_1, …)} \\
\left(d\!\left(\omega\right)\right)\!\left(X,\, Y\right) &\to X\!\left(\omega\!\left(Y\right)\right) - Y\!\left(\omega\!\left(X\right)\right) - \omega\!\left([X,Y]_{VF}\right) && \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)} \\
\left(d\!\left(\iota_X\!\left(\omega\right)\right)\right)\!\left(Y\right) &\to Y\!\left(\iota_X\!\left(\omega\right)\right) && \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)} \\
Y\!\left(\iota_X\!\left(\omega\right)\right) &\to Y\!\left(\omega\!\left(X\right)\right) && \text{[bare \ensuremath{\iota}\_X(\ensuremath{\omega}) inside Act(D, \_) → Act(D, MultiEval(\ensuremath{\omega}, X))]\,(axiom)}\;\text{--- apply axiom: bare \ensuremath{\iota}\_X(\ensuremath{\omega}) inside Act(D, \_) → Act(D, MultiEval(\ensuremath{\omega}, X))} \\
\left(L_X\!\left(\omega\right)\right)\!\left(Y\right) &\to X\!\left(\omega\!\left(Y\right)\right) - \omega\!\left([X,Y]_{VF}\right) && \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)} \\
\left(\left(X\!\left(\omega\!\left(Y\right)\right) - Y\!\left(\omega\!\left(X\right)\right) - \omega\!\left([X,Y]_{VF}\right)\right) + Y\!\left(\omega\!\left(X\right)\right)\right) - \left(X\!\left(\omega\!\left(Y\right)\right) - \omega\!\left([X,Y]_{VF}\right)\right) &\to 0 && \text{[simplify]}\;\text{--- canonical-form pipeline (intra-loop)}
\end{align*}

## 4. İlişki 4: $[\mathcal{L}_X, \iota_Y] = \iota_{[X,Y]_{VF}}$

$$
\mathcal{L}_X \iota_Y - \iota_Y \mathcal{L}_X \;=\; \iota_{[X,Y]_{VF}}.
$$

### 4.1. 0-form $f$ üzerinde

$f$ üzerinde her terim 0; eşitlik triviyal — engine bunu Cartan
magic + $\iota$-on-0-form rotasıyla buluyor.

In [12]:
lhs = Sum(Act(lie_derivative(X), Act(interior(Y), f)),
           Neg(Act(interior(Y), Act(lie_derivative(X), f))))
rhs = Act(interior(lie_bracket_vf(X, Y)), f)
chain = prove("(0-form) [L_X, ι_Y] f = ι_[X,Y] f", lhs, rhs, engine_zf)
display_chain(chain)

(0-form) [L_X, ι_Y] f = ι_[X,Y] f → 12 adımda kapandı


\begin{align*}
\iota_Y\!\left(f\right) &\to 0 && \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_X(f) = 0 on 0-forms} \\
L_X\!\left(0\right) &\to \left(d \, \iota_X\right)\!\left(0\right) + \left(\iota_X \, d\right)\!\left(0\right) && \text{[L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)]\,(axiom)}\;\text{--- apply axiom: L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)} \\
L_X\!\left(f\right) &\to \left(d \, \iota_X\right)\!\left(f\right) + \left(\iota_X \, d\right)\!\left(f\right) && \text{[L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)]\,(axiom)}\;\text{--- apply axiom: L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)} \\
\iota_[X,Y]_{VF}\!\left(f\right) &\to 0 && \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_X(f) = 0 on 0-forms} \\
\left(\left(\left(d \, \iota_X\right)\!\left(0\right) + \left(\iota_X \, d\right)\!\left(0\right)\right) - \iota_Y\!\left(\left(d \, \iota_X\right)\!\left(f\right) + \left(\iota_X \, d\right)\!\left(f\right)\right)\right) - 0 &\to \left(\left(0 + 0\right) - \left(\iota_Y\!\left(d\!\left(\iota_X\!\left(f\right)\right)\right) + \iota_Y\!\left(\iota_X\!\left(d\!\left(f\right)\right)\right)\right)\right) - 0 && \text{[product-rule]}\;\text{--- graded Leibniz + linearity} \\
\left(\left(0 + 0\right) - \left(\iota_Y\!\left(d\!\left(\iota_X\!\left(f\right)\right)\right) + \iota_Y\!\left(\iota_X\!\left(d\!\left(f\right)\right)\right)\right)\right) - 0 &\to -\iota_Y\!\left(d\!\left(\iota_X\!\left(f\right)\right)\right) - \iota_Y\!\left(\iota_X\!\left(d\!\left(f\right)\right)\right) && \text{[simplify]}\;\text{--- canonical-form pipeline (intra-loop)} \\
\iota_X\!\left(f\right) &\to 0 && \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_X(f) = 0 on 0-forms} \\
\iota_Y\!\left(d\!\left(0\right)\right) &\to Y\!\left(0\right) && \text{[\ensuremath{\iota}\_X(df) = X(f)]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_X(df) = X(f)} \\
\iota_X\!\left(d\!\left(f\right)\right) &\to X\!\left(f\right) && \text{[\ensuremath{\iota}\_X(df) = X(f)]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_X(df) = X(f)} \\
\iota_Y\!\left(X\!\left(f\right)\right) &\to 0 && \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_X(f) = 0 on 0-forms} \\
-Y\!\left(0\right) - 0 &\to -0 - 0 && \text{[product-rule]}\;\text{--- graded Leibniz + linearity} \\
-0 - 0 &\to 0 && \text{[simplify]}\;\text{--- canonical-form pipeline (intra-loop)}
\end{align*}

### 4.2. 1-form $\omega$ üzerinde, $Y$ ile değerlendir

Her iki taraf 0-form skalere düşer; ifadeleri açıkça `multi_eval(\omega, Y)` $\equiv \omega(Y)$ formunda yazıyoruz, böylece intrinsik kurallar fire eder. Sol taraftaki $\mathcal{L}_X(\iota_Y \omega) = \mathcal{L}_X(\omega(Y))$ 0-form skalere düştüğü için *flow*-mode $\mathcal{L}_X$ kullanıyoruz; bu, $\mathcal{L}_X(\omega(Y)) \to X(\omega(Y))$ adımını açar.

In [13]:
from jacopy.calculus.lie_derivative import LieDerivative

L_X_flow = LieDerivative(X, definition="flow")  # 0-form skalerine düşmek için
L_X_full = lie_derivative(X)                    # form-tarafı için cartan-mode

lhs = Sum(
    Act(L_X_flow, multi_eval(omega, Y)),                  # L_X(ω(Y))
    Neg(multi_eval(Act(L_X_full, omega), Y)),             # (L_X ω)(Y)
)
rhs = multi_eval(omega, lie_bracket_vf(X, Y))              # ω([X,Y])
chain = prove("(1-form, eval Y) L_X(ω(Y)) − (L_X ω)(Y) = ω([X,Y])",
              lhs, rhs, engine_of)
display_chain(chain)

(1-form, eval Y) L_X(ω(Y)) − (L_X ω)(Y) = ω([X,Y]) → 3 adımda kapandı


\begin{align*}
L_X\!\left(\omega\!\left(Y\right)\right) &\to X\!\left(\omega\!\left(Y\right)\right) && \text{[L\_X(f) = X(f) on 0-forms (flow)]\,(axiom)}\;\text{--- apply axiom: L\_X(f) = X(f) on 0-forms (flow)} \\
\left(L_X\!\left(\omega\right)\right)\!\left(Y\right) &\to X\!\left(\omega\!\left(Y\right)\right) - \omega\!\left([X,Y]_{VF}\right) && \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)} \\
\left(X\!\left(\omega\!\left(Y\right)\right) - \left(X\!\left(\omega\!\left(Y\right)\right) - \omega\!\left([X,Y]_{VF}\right)\right)\right) - \omega\!\left([X,Y]_{VF}\right) &\to 0 && \text{[simplify]}\;\text{--- canonical-form pipeline (intra-loop)}
\end{align*}

## 5. İlişki 5: $[d, \mathcal{L}_X] = 0$

$$
d \, \mathcal{L}_X \;=\; \mathcal{L}_X \, d.
$$

### 5.1. 0-form $f$ üzerinde

$f$ için $\mathcal{L}_X f = X(f)$, $df$ bir 1-form;
*flow* aksiyomu $\mathcal{L}_X \circ d = d \circ \mathcal{L}_X$
direkt çalışıyor.

In [14]:
lhs = Sum(Act(default_d, Act(lie_derivative(X), f)),
           Neg(Act(lie_derivative(X), Act(default_d, f))))
chain = prove("(0-form) (d L_X − L_X d) f = 0", lhs, Integer(0), engine_zf)
display_chain(chain)

(0-form) (d L_X − L_X d) f = 0 → 9 adımda kapandı


\begin{align*}
L_X\!\left(f\right) &\to \left(d \, \iota_X\right)\!\left(f\right) + \left(\iota_X \, d\right)\!\left(f\right) && \text{[L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)]\,(axiom)}\;\text{--- apply axiom: L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)} \\
L_X\!\left(d\!\left(f\right)\right) &\to \left(d \, \iota_X\right)\!\left(d\!\left(f\right)\right) + \left(\iota_X \, d\right)\!\left(d\!\left(f\right)\right) && \text{[L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)]\,(axiom)}\;\text{--- apply axiom: L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)} \\
\left(d\!\left(\left(d \, \iota_X\right)\!\left(f\right) + \left(\iota_X \, d\right)\!\left(f\right)\right) - \left(\left(d \, \iota_X\right)\!\left(d\!\left(f\right)\right) + \left(\iota_X \, d\right)\!\left(d\!\left(f\right)\right)\right)\right) - 0 &\to \left(\left(d\!\left(d\!\left(\iota_X\!\left(f\right)\right)\right) + d\!\left(\iota_X\!\left(d\!\left(f\right)\right)\right)\right) - \left(d\!\left(\iota_X\!\left(d\!\left(f\right)\right)\right) + \iota_X\!\left(d\!\left(d\!\left(f\right)\right)\right)\right)\right) - 0 && \text{[product-rule]}\;\text{--- graded Leibniz + linearity} \\
\left(\left(d\!\left(d\!\left(\iota_X\!\left(f\right)\right)\right) + d\!\left(\iota_X\!\left(d\!\left(f\right)\right)\right)\right) - \left(d\!\left(\iota_X\!\left(d\!\left(f\right)\right)\right) + \iota_X\!\left(d\!\left(d\!\left(f\right)\right)\right)\right)\right) - 0 &\to d\!\left(d\!\left(\iota_X\!\left(f\right)\right)\right) - \iota_X\!\left(d\!\left(d\!\left(f\right)\right)\right) && \text{[simplify]}\;\text{--- canonical-form pipeline (intra-loop)} \\
\iota_X\!\left(f\right) &\to 0 && \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_X(f) = 0 on 0-forms} \\
d\!\left(d\!\left(0\right)\right) &\to 0 && \text{[d² = 0]\,(axiom)}\;\text{--- apply axiom: d² = 0} \\
d\!\left(d\!\left(f\right)\right) &\to 0 && \text{[d² = 0]\,(axiom)}\;\text{--- apply axiom: d² = 0} \\
\iota_X\!\left(0\right) &\to 0 && \text{[\ensuremath{\iota}\_X(f) = 0 on 0-forms]\,(axiom)}\;\text{--- apply axiom: \ensuremath{\iota}\_X(f) = 0 on 0-forms} \\
0 - 0 &\to 0 && \text{[simplify]}\;\text{--- canonical-form pipeline (intra-loop)}
\end{align*}

### 5.2. 1-form $\omega$ üzerinde, $(Y, Z)$ ile değerlendir

1-form $\omega$ için fark 2-form'dur; $(Y, Z)$ ile
değerlendirip Koszul açılımı + commutator + Jacobi adımları üzerinden
sıfırlatıyoruz.

In [15]:
lhs = Sum(multi_eval(Act(default_d, Act(lie_derivative(X), omega)), Y, Z),
           Neg(multi_eval(Act(lie_derivative(X), Act(default_d, omega)), Y, Z)))
chain = prove("(1-form, eval Y,Z) ((d L_X − L_X d) ω)(Y, Z) = 0",
              lhs, Integer(0), engine_of)
display_chain(chain)

(1-form, eval Y,Z) ((d L_X − L_X d) ω)(Y, Z) = 0 → 15 adımda kapandı


\begin{align*}
\left(d\!\left(L_X\!\left(\omega\right)\right)\right)\!\left(Y,\, Z\right) &\to Y\!\left(\left(L_X\!\left(\omega\right)\right)\!\left(Z\right)\right) - Z\!\left(\left(L_X\!\left(\omega\right)\right)\!\left(Y\right)\right) - \left(L_X\!\left(\omega\right)\right)\!\left([Y,Z]_{VF}\right) && \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)} \\
\left(L_X\!\left(\omega\right)\right)\!\left(Z\right) &\to X\!\left(\omega\!\left(Z\right)\right) - \omega\!\left([X,Z]_{VF}\right) && \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)} \\
\left(L_X\!\left(\omega\right)\right)\!\left(Y\right) &\to X\!\left(\omega\!\left(Y\right)\right) - \omega\!\left([X,Y]_{VF}\right) && \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)} \\
\left(L_X\!\left(\omega\right)\right)\!\left([Y,Z]_{VF}\right) &\to X\!\left(\omega\!\left([Y,Z]_{VF}\right)\right) - \omega\!\left([X,[Y,Z]_{VF}]_{VF}\right) && \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)} \\
\left(L_X\!\left(d\!\left(\omega\right)\right)\right)\!\left(Y,\, Z\right) &\to X\!\left(\left(d\!\left(\omega\right)\right)\!\left(Y,\, Z\right)\right) - \left(d\!\left(\omega\right)\right)\!\left([X,Y]_{VF},\, Z\right) - \left(d\!\left(\omega\right)\right)\!\left(Y,\, [X,Z]_{VF}\right) && \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)} \\
\left(d\!\left(\omega\right)\right)\!\left(Y,\, Z\right) &\to Y\!\left(\omega\!\left(Z\right)\right) - Z\!\left(\omega\!\left(Y\right)\right) - \omega\!\left([Y,Z]_{VF}\right) && \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)} \\
\left(d\!\left(\omega\right)\right)\!\left([X,Y]_{VF},\, Z\right) &\to [X,Y]_{VF}\!\left(\omega\!\left(Z\right)\right) - Z\!\left(\omega\!\left([X,Y]_{VF}\right)\right) - \omega\!\left([[X,Y]_{VF},Z]_{VF}\right) && \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\en

## 6. İlişki 6: $d^2 = 0$

$$
d \circ d \;=\; 0.
$$

### 6.1. 0-form $f$ üzerinde

$f$ üzerinde tek bir aksiyom: $d(df) = 0$.

In [16]:
lhs = Act(default_d, Act(default_d, f))
chain = prove("(0-form) d(df) = 0", lhs, Integer(0), engine_zf)
display_chain(chain)

(0-form) d(df) = 0 → 2 adımda kapandı


\begin{align*}
d\!\left(d\!\left(f\right)\right) &\to 0 && \text{[d² = 0]\,(axiom)}\;\text{--- apply axiom: d² = 0} \\
0 - 0 &\to 0 && \text{[simplify]}\;\text{--- canonical-form pipeline (intra-loop)}
\end{align*}

### 6.2. 1-form $\omega$ üzerinde, $(X, Y, Z)$ ile değerlendir

1-form $\omega$ için $d(d\omega)$ 3-form, $(X, Y, Z)$ ile
değerlendiriyoruz. Açılım iki kat Koszul + her bracket çiftinde
$X(Y(g)) - Y(X(g)) = [X,Y]_{VF}(g)$ commutator + üç-bracket Jacobi
zinciri ile kapanıyor.

In [17]:
lhs = multi_eval(Act(default_d, Act(default_d, omega)), X, Y, Z)
chain = prove("(1-form, eval X,Y,Z) (d(dω))(X, Y, Z) = 0",
              lhs, Integer(0), engine_of)
display_chain(chain)

(1-form, eval X,Y,Z) (d(dω))(X, Y, Z) = 0 → 15 adımda kapandı


\begin{align*}
\left(d\!\left(d\!\left(\omega\right)\right)\right)\!\left(X,\, Y,\, Z\right) &\to X\!\left(\left(d\!\left(\omega\right)\right)\!\left(Y,\, Z\right)\right) - Y\!\left(\left(d\!\left(\omega\right)\right)\!\left(X,\, Z\right)\right) + Z\!\left(\left(d\!\left(\omega\right)\right)\!\left(X,\, Y\right)\right) - \left(d\!\left(\omega\right)\right)\!\left([X,Y]_{VF},\, Z\right) + \left(d\!\left(\omega\right)\right)\!\left([X,Z]_{VF},\, Y\right) - \left(d\!\left(\omega\right)\right)\!\left([Y,Z]_{VF},\, X\right) && \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)} \\
\left(d\!\left(\omega\right)\right)\!\left(Y,\, Z\right) &\to Y\!\left(\omega\!\left(Z\right)\right) - Z\!\left(\omega\!\left(Y\right)\right) - \omega\!\left([Y,Z]_{VF}\right) && \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)} \\
\left(d\!\left(\omega\right)\right)\!\left(X,\, Z\right) &\to X\!\left(\omega\!\left(Z\right)\right) - Z\!\left(\omega\!\left(X\right)\right) - \omega\!\left([X,Z]_{VF}\right) && \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)} \\
\left(d\!\left(\omega\right)\right)\!\left(X,\, Y\right) &\to X\!\left(\omega\!\left(Y\right)\right) - Y\!\left(\omega\!\left(X\right)\right) - \omega\!\left([X,Y]_{VF}\right) && \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)} \\
\left(d\!\left(\omega\right)\right)\!\left([X,Y]_{VF},\, Z\right) &\to [X,Y]_{VF}\!\left(\omega\!\left(Z\right)\right) - Z\!\left(\omega\!\left([X,Y]_{VF}\right)\right) - \omega\!\left([[X,Y]_{VF},Z]_{VF}\right) && \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)} \\
\left(d\!\left(\omega\right)\right)\!\left([X,Z]_{VF},\, Y\right) &\to [X,Z]_{VF}\!\left(\omega\!\left(Y\right)\right) - Y\!\left(\omega\!\left([X,Z]_{VF}\right)\right) - \omega\!\left([[X,Z]_{VF},Y]_{VF}\right) && \text{[d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: d intrinsic (Koszul): (d\ensuremath{\omega})(X\_0, …, X\_p) = \ensuremath{\Sigma} ±X\_i(\ensuremath{\omega}(…,hat\_i,…)) + \ensuremath{\Sigma} ±\ensuremath{\omega}([X\_i,X\_j]\_VF, …)} \\
\left(d\!\left(\omega

## 7. İlişki 7: $[\mathcal{L}_X, \mathcal{L}_Y] = \mathcal{L}_{[X,Y]_{VF}}$

$$
\mathcal{L}_X \mathcal{L}_Y - \mathcal{L}_Y \mathcal{L}_X
   \;=\; \mathcal{L}_{[X,Y]_{VF}}.
$$

### 7.1. 0-form $f$ üzerinde

$f$ üzerinde her iki taraf skaler: $X(Y(f)) - Y(X(f)) =
[X,Y]_{VF}(f)$. Tam olarak `VfActCommutatorDefinition`'ın yaptığı iş.

In [18]:
lhs = Sum(Act(lie_derivative(X), Act(lie_derivative(Y), f)),
           Neg(Act(lie_derivative(Y), Act(lie_derivative(X), f))))
rhs = Act(lie_derivative(lie_bracket_vf(X, Y)), f)
chain = prove("(0-form) [L_X, L_Y] f = L_[X,Y] f", lhs, rhs, engine_zf)
display_chain(chain)

(0-form) [L_X, L_Y] f = L_[X,Y] f → 30 adımda kapandı


\begin{align*}
L_Y\!\left(f\right) &\to \left(d \, \iota_Y\right)\!\left(f\right) + \left(\iota_Y \, d\right)\!\left(f\right) && \text{[L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)]\,(axiom)}\;\text{--- apply axiom: L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)} \\
L_X\!\left(\left(d \, \iota_Y\right)\!\left(f\right) + \left(\iota_Y \, d\right)\!\left(f\right)\right) &\to \left(d \, \iota_X\right)\!\left(\left(d \, \iota_Y\right)\!\left(f\right) + \left(\iota_Y \, d\right)\!\left(f\right)\right) + \left(\iota_X \, d\right)\!\left(\left(d \, \iota_Y\right)\!\left(f\right) + \left(\iota_Y \, d\right)\!\left(f\right)\right) && \text{[L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)]\,(axiom)}\;\text{--- apply axiom: L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)} \\
L_X\!\left(f\right) &\to \left(d \, \iota_X\right)\!\left(f\right) + \left(\iota_X \, d\right)\!\left(f\right) && \text{[L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)]\,(axiom)}\;\text{--- apply axiom: L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)} \\
L_Y\!\left(\left(d \, \iota_X\right)\!\left(f\right) + \left(\iota_X \, d\right)\!\left(f\right)\right) &\to \left(d \, \iota_Y\right)\!\left(\left(d \, \iota_X\right)\!\left(f\right) + \left(\iota_X \, d\right)\!\left(f\right)\right) + \left(\iota_Y \, d\right)\!\left(\left(d \, \iota_X\right)\!\left(f\right) + \left(\iota_X \, d\right)\!\left(f\right)\right) && \text{[L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)]\,(axiom)}\;\text{--- apply axiom: L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)} \\
L_[X,Y]_{VF}\!\left(f\right) &\to \left(d \, \iota_[X,Y]_{VF}\right)\!\left(f\right) + \left(\iota_[X,Y]_{VF} \, d\right)\!\left(f\right) && \text{[L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)]\,(axiom)}\;\text{--- apply axiom: L\_X := d\ensuremath{\circ}\ensuremath{\iota}\_X + \ensuremath{\iota}\_X\ensuremath{\circ}d (Cartan definition)} \\
\left(\left(\left(d \, \iota_X\right)\!\left(\left(d \, \iota_Y\right)\!\left(f\right) + \left(\iota_Y \, d\right)\!\left(f\right)\right) + \left(\iota_X \, d\right)\!\left(\left(d \, \iota_Y\right)\!\left(f\right) + \left(\iota_Y \, d\right)\!\left(f\right)\right)\right) - \left(\left(d \, \iota_Y\right)\!\left(\left(d \, \iota_X\right)\!\left(f\right) + \left(\iota_X \, d\right)\!\left(f\right)\right) + \left(\iota_Y \, d\right)\!\left(\left(d \, \iota_X\right)\!\left(f\right) + \left(\iota_X \, d\right)\!\left(f\right)\right)\right)\right) - \left(\left(d \, \iota_[X,Y]_{VF}\right)\!\left(f\right) + \left(\iota_[X,Y]_{VF} \, d\right)\!\left(f\right)\right) &\to \left(\left(\left(d\!\left(\iota_X\!\left(d\!\left(\iota_Y\!\left(f\right)\right)\right)\right) + d\!\left(\iota_X\!\left(\iota_Y\!\left(d\!\left(f\right)\right)\right)\right)\right) + \left(\iota_X\!\left(d\!\left(d\!\left(\iota_Y\!\left(f\right)\right)\right)\right) + \iota_X\!\left(d\!\left(\iota_Y\!\left(d\!\left(f\right)\right)\right)\right)\right)\right) - \left(\left(d\!\left(\iota_Y\!\left(d\!\left(\iota_X\!\left(f\right)\right)\right)\right) + d\!\left(\iota_Y\!\left(\iota_X\!\left(d\!\left(f\right)\right)\right)\right)\right) + \left(\iota_Y\!\left(d\!\left(d\!\left(\iota_X\!\left(f\right)\right)\right)\right) + \iota_Y\!\left(d\!\left(\iota_X\!\left(d\!\left(f\right)\right)\right)\right)\right)\right)\right) - \left(d\!\left(\iota_[X,Y]_{VF}\!\left(f\right)\right) + \iota_[X,Y]_{VF}\!\left(d\!\left(f\right)\right)\right) && \text{[product

### 7.2. 1-form $\omega$ üzerinde, $Z$ ile değerlendir

1-form $\omega$ için fark 1-form'dur; $Z$ ile değerlendirip
intrinsik açılım + commutator/Jacobi zinciri üzerinden tek pipeline'da
kapatıyoruz.

In [19]:
lhs = Sum(multi_eval(Act(lie_derivative(X), Act(lie_derivative(Y), omega)), Z),
           Neg(multi_eval(Act(lie_derivative(Y), Act(lie_derivative(X), omega)), Z)))
rhs = multi_eval(Act(lie_derivative(lie_bracket_vf(X, Y)), omega), Z)
chain = prove("(1-form, eval Z) ([L_X, L_Y] ω)(Z) = (L_[X,Y] ω)(Z)",
              lhs, rhs, engine_of)
display_chain(chain)

(1-form, eval Z) ([L_X, L_Y] ω)(Z) = (L_[X,Y] ω)(Z) → 12 adımda kapandı


\begin{align*}
\left(L_X\!\left(L_Y\!\left(\omega\right)\right)\right)\!\left(Z\right) &\to X\!\left(\left(L_Y\!\left(\omega\right)\right)\!\left(Z\right)\right) - \left(L_Y\!\left(\omega\right)\right)\!\left([X,Z]_{VF}\right) && \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)} \\
\left(L_Y\!\left(\omega\right)\right)\!\left(Z\right) &\to Y\!\left(\omega\!\left(Z\right)\right) - \omega\!\left([Y,Z]_{VF}\right) && \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)} \\
\left(L_Y\!\left(\omega\right)\right)\!\left([X,Z]_{VF}\right) &\to Y\!\left(\omega\!\left([X,Z]_{VF}\right)\right) - \omega\!\left([Y,[X,Z]_{VF}]_{VF}\right) && \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)} \\
\left(L_Y\!\left(L_X\!\left(\omega\right)\right)\right)\!\left(Z\right) &\to Y\!\left(\left(L_X\!\left(\omega\right)\right)\!\left(Z\right)\right) - \left(L_X\!\left(\omega\right)\right)\!\left([Y,Z]_{VF}\right) && \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)} \\
\left(L_X\!\left(\omega\right)\right)\!\left(Z\right) &\to X\!\left(\omega\!\left(Z\right)\right) - \omega\!\left([X,Z]_{VF}\right) && \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)} \\
\left(L_X\!\left(\omega\right)\right)\!\left([Y,Z]_{VF}\right) &\to X\!\left(\omega\!\left([Y,Z]_{VF}\right)\right) - \omega\!\left([X,[Y,Z]_{VF}]_{VF}\right) && \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)} \\
\left(L_[X,Y]_{VF}\!\left(\omega\right)\right)\!\left(Z\right) &\to [X,Y]_{VF}\!\left(\omega\!\left(Z\right)\right) - \omega\!\left([[X,Y]_{VF},Z]_{VF}\right) && \text{[L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)]\,(axiom)}\;\text{--- apply axiom: L\_X intrinsic: (L\_X \ensuremath{\omega})(Y\_1, …) = X(\ensuremath{\omega}(Y\_1, …)) − \ensuremath{\Sigma} \ensuremath{\omega}(…, [X, Y\_i]\_VF, …)} \\
\left(\left(X\!\left(Y\!\left(\omega\!\left(Z\right)\right) - \omega\!\left([Y,Z]_{VF}\right)\right) - \left(Y\!\left(\omega\!\left([X,Z]_{VF}\right)\right) - \omega\!\left([Y,[X,Z]_{VF}]_{VF}\right)\right)\right) - \left(Y\!\left(X\!\left(\omega\!\left(Z\right)\right) - \omega\!\left([X,Z]_{VF}\right)\right) - \left(X\!\left(\omega\!\left([Y,Z]_{VF}\right)\right) - \omega\!\left([X,[Y,Z]

## Sonuç

Yedi Cartan ilişkisinin tamamı önce 0-form $f$ üzerinde, sonra 1-form
$\omega$ üzerinde tek bir
`prove_intrinsic_equivalence` çağrısı + uygun engine bileşimi ile
syntactic olarak kapandı.

| # | İlişki | 0-form | 1-form |
|---|---|---|---|
| 1 | $\iota \iota$ anti-commute | $\iota_X(f) = 0$ ile triviyal | $\iota_X(\iota_Y\omega) = 0$ via 0-form çöküş |
| 2 | Cartan magic | $\iota_X(df) + d(\iota_X f) = X(f)$ | Koszul açılım + bracket iptali |
| 3 | $[\mathcal{L}_X, \iota_Y] = \iota_{[X,Y]}$ | her iki taraf 0 | $\iota$'lar `multi_eval(ω, Y)` ile yazılır; *flow* $\mathcal{L}_X$ skalere iner |
| 4 | $[d, \mathcal{L}_X] = 0$ | flow aksiyomu | 2 katlı Koszul + Jacobi |
| 5 | $d^2 = 0$ | $d(df) = 0$ aksiyomu | 3-form değerlendirmesi + commutator/Jacobi |
| 6 | $[\mathcal{L}_X, \mathcal{L}_Y] = \mathcal{L}_{[X,Y]}$ | `VfActCommutator` | intrinsik açılım + commutator |

Bütün ispatlar `display_chain` ile LaTeX olarak okunabilir; her adımın
hangi `Definition`'dan geldiği `ProofStep.rule` alanında saklanıyor.